In [37]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:10pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:80px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:2px;}
table.dataframe{font-size:10pt;} 
</style>
"""))

<font size="6" color="green"><b>ch15. 데이터베이스 연동</b></font>
# 1절.  SQLite 데이터 베이스 연결

- SQLite 데이터베이스는 별도의 DBMS없이 SQL을 이용해서 DB액세스할 수 있도록 만든 간단한 디스크시반 DB제공
- C라이브러리
- SQLite는 프로토타입을 만들 때 사용
- 프로젝트 단계 : 분석 → 설계 → 구현 → 테스트 → 고객에게 배포 → 유지보수
        - 프로토타입(SQLite) 시제품(구현 후 반양산직전) 완제품(Oracle, MySQL, MasriaDB,
        PostgresSQL, MSSQL, 아마존aroraDB,...)
        
- [DB Browser for SQLite] 
<br>
(https://sqlitebrowser.org/)에서 "DB Browser for SQLite - .zip (no installer) for 64-bit Windows" 다운로드 후 압축 풀기
    
## 1.1 SQLite browser 설치 및 sqlite3패키지 load

In [3]:
import sqlite3
sqlite3.sqlite_version

'3.40.1'

## 1.2 데이터베이스 연결

- 데이터베이스 연결 객체 → 커서객체(SQL전송 및 결과 받는 객체) → 원하는 로직 수행 → 커서객체 해제 → DB연결 객체 해제(close)
- SQLite로 DB 연결객체 생성시, DB파일이 있으면 연결, DB파일이 없으면 빈 DB파일 생성

### ▼ DB연결 (여기서 에러가 날 경우 VC_redist.x64.exe 설치)

In [5]:
conn = sqlite3.connect('data/ch15_example.db')
conn

### ▼ 커서 객체 생성하기 : 커서는 SQL문 실생시키고, 결과를 받는 객체

In [6]:
cursor = conn.cursor()
cursor

In [10]:
cursor.execute('''
    CREATE TABLE MEMBER (
        NAME TEXT,
        AGE INT,
        EMAIL TEXT
    )
''')

In [11]:
cursor.execute('DROP TABLE MEMBER')

### ▼ 원하는 로직 수행

In [14]:
sql = 'INSERT INTO MEMBER VALUES (\'마동석\', 25, \'h@h.com\')'
cursor.execute(sql) # sql 전송
print('insert, update, delete문의 수행 결과 행수 :', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('고길동', 55, 'g@h.com')"
cursor.execute(sql)
print('수행 결과 행수 :', cursor.rowcount)
sql = "INSERT INTO MEMBER VALUES ('하석진', 36, 'h@h.com')"
cursor.execute(sql)
print('수행 결과 행수 :', cursor.rowcount)

insert, update, delete문의 수행 결과 행수 : 1
수행 결과 행수 : 1
수행 결과 행수 : 1


In [15]:
conn.commit() # 反. conn.rollback()dML에서만 commit이나 rollback

### # SELECT SQL문 전송

In [17]:
cursor.execute("SELECT * FROM MEMBER ODERE BY AGE")

In [19]:
# INSERT, UPDATE, DELETE문 실행결과 : cursor.rowcount
# SELECT문 실행결과를 받는 함수들
    # fetchone(): 결과를 한행씩 받을 때 (튜플)
    # fetchall(): 결과를 모두 받을 때 (튜플 list)
    # fetchmany(n): 결과를 n행 받을 때 (튜플 list)
cursor.fetchall()

[('홍길동', 25, 'h@h.com'),
 ('홍길동', 25, 'h@h.com'),
 ('하석진', 36, 'h@h.com'),
 ('고길동', 55, 'g@h.com')]

In [20]:
cursor.fetchall() # 한번 소요된 cursor객체는 다시 fetch할 수 없음.

[]

In [22]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchall()
members

[('홍길동', 25, 'h@h.com'),
 ('홍길동', 25, 'h@h.com'),
 ('하석진', 36, 'h@h.com'),
 ('고길동', 55, 'g@h.com')]

### # 한줄씩 읽어서 dict list에 append

In [24]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    member =cursor.fetchone()#SQL문 수행결과문 한줄씩가져오기
    if member is None:
        break 
    members.append({'name':member[0], 'age':member[1], 'email':member[2]})
members

[{'name': '홍길동', 'age': 25, 'email': 'h@h.com'},
 {'name': '홍길동', 'age': 25, 'email': 'h@h.com'},
 {'name': '하석진', 'age': 36, 'email': 'h@h.com'},
 {'name': '고길동', 'age': 55, 'email': 'g@h.com'}]

In [25]:
class Member:
    'Member 테이블의 내용을 받을 객체 타입'
    def __init__(self,name, age, email):
        self.name = name
        self.age = age
        self.email = email
    def __str__(self):
        return "{}\t{}\t{}".format(self.name, self.age, self.email)
m = Member('홍길동', 25, 'h@h.com')
print(m)

홍길동	25	h@h.com


In [28]:
dbmember = ('홍길동', 25, 'h@h.com')
m = Member(*dbmember)
print(m)

홍길동	25	h@h.com


### # 한줄씩 읽어서 객체 list에 append

In [30]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = []
while True:
    dbmember = cursor.fetchone()
    if dbmember is None:
        break
    member = Member(*dbmember)
    members.append(member)
for mem in members:
    print(mem)

홍길동	25	h@h.com
홍길동	25	h@h.com
하석진	36	h@h.com
고길동	55	g@h.com


### # 최상위 n행 읽어오기

In [31]:
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchmany(2)
members

[('홍길동', 25, 'h@h.com'), ('홍길동', 25, 'h@h.com')]

In [32]:
cursor.close()
conn.close()

## 1.3 SQL 구문에 파라미터 사용하기
- qmark(DB에 따라 불가한 경우가 있음)
- named(추천)

In [33]:
conn = sqlite3.connect('Data/ch15_example.db')
cursor = conn.cursor()
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN ('고길동','홍길동')")
cursor.fetchall()

[('홍길동', 25, 'h@h.com'), ('홍길동', 25, 'h@h.com'), ('고길동', 55, 'g@h.com')]

### # 파라미터 사용하기
#### - qmark 방법 이용

In [35]:
name1 = input('검색할 이름1 : ')
name2 = input('검색할 이름2 : ')
# cursor.execute(f"SELECT * FROM MEMBER WHERE NAME IN('{name1}', '{name2}')")
# cursor.execute("SELECT * FROM MEMBER WHERE NAME IN(:name1, :name2)", {'name1':name1, 'name2':name2})
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN(?,?)", (name1, name2))
cursor.fetchall()

검색할 이름1 : 하석진
검색할 이름2 : 마동석


[('하석진', 36, 'h@h.com')]

#### - named 방법

In [39]:
name = input('회원가입할 이름은?')
try:
    age = int(input('나이는?(꼭 숫자로)'))
except:
    print('유효하지 않은 나이를 입력할 경우 0로 초기화합니다.')
    age=0
email = input('이메일은?')
cursor.execute( "INSERT INTO MEMBER VALUES (:name, :age, :email)"
               , {'name':name, 'age':age, 'email':email}) # sql문 전송
conn.commit()
print('수행 결과 행수 :', cursor.rowcount)

회원가입할 이름은?하길동
나이는?(꼭 숫자로)이팔
유효하지 않은 나이를 입력할 경우 0로 초기화합니다.
이메일은?l@h.com
수행 결과 행수 : 1


In [40]:
cursor.close()
conn.close()

## 2절. 오라클 데이터베이스 연결
- pip install cx_oracle(11g까지), pip install oracledb(12버전부터)

In [41]:
import cx_Oracle
cx_Oracle.__version__

'8.3.0'

###  ▼ conn 얻어오는 방법1

In [43]:
oracal_dsn = cx_Oracle.makedsn(host="localhost", port=1521, sid='xe')
# conn = cx_Oracle.connect(user='scott', password='tiger', dsn=oracal_dsn)
conn = cx_Oracle.connect('scott', 'tiger', oracal_dsn)
conn.close()

###  ▼ conn 얻어오는 방법2 : (여기서 에러가 날 경우 VC_redist.x64.exe 설치)

In [44]:
# conn = cx_Oracle.connect(user='scott', password='tiger', dsn='localhost:1521/xe')
conn = cx_Oracle.connect('scott', 'tiger', 'localhost:1521/xe')
conn

<cx_Oracle.Connection to scott@localhost:1521/xe>

###  ▼cursor 객체 생성하고 sql문 전송 & 결과 받기

In [46]:
cursor = conn.cursor()
sql = "SELECT EMPNO NO, ENAME, JOB, MGR, HIREDATE, SAL, COMM, DEPTNO FROM EMP"
cursor.execute(sql)
emps = cursor.fetchall()

In [47]:
for emp in emps:
    print(emps)

[(7369, 'SMITH', 'CLERK', 7902, datetime.datetime(1980, 12, 17, 0, 0), 800.0, None, 20), (7499, 'ALLEN', 'SALESMAN', 7698, datetime.datetime(1981, 2, 20, 0, 0), 1600.0, 300.0, 30), (7521, 'WARD', 'SALESMAN', 7698, datetime.datetime(1981, 2, 22, 0, 0), 1250.0, 500.0, 30), (7566, 'JONES', 'MANAGER', 7839, datetime.datetime(1981, 4, 2, 0, 0), 2975.0, None, 20), (7654, 'MARTIN', 'SALESMAN', 7698, datetime.datetime(1981, 9, 28, 0, 0), 1250.0, 1400.0, 30), (7698, 'BLAKE', 'MANAGER', 7839, datetime.datetime(1981, 5, 1, 0, 0), 2850.0, None, 30), (7782, 'CLARK', 'MANAGER', 7839, datetime.datetime(1981, 6, 9, 0, 0), 2450.0, None, 10), (7788, 'SCOTT', 'ANALYST', 7566, datetime.datetime(1982, 12, 9, 0, 0), 3000.0, None, 20), (7839, 'KING', 'PRESIDENT', None, datetime.datetime(1981, 11, 17, 0, 0), 5000.0, None, 10), (7844, 'TURNER', 'SALESMAN', 7698, datetime.datetime(1981, 9, 8, 0, 0), 1500.0, 0.0, 30), (7876, 'ADAMS', 'CLERK', 7788, datetime.datetime(1983, 1, 12, 0, 0), 1100.0, None, 20), (7900, 

In [49]:
cursor.description

[('NO', <cx_Oracle.DbType DB_TYPE_NUMBER>, 5, None, 4, 0, 0),
 ('ENAME', <cx_Oracle.DbType DB_TYPE_VARCHAR>, 10, 10, None, None, 1),
 ('JOB', <cx_Oracle.DbType DB_TYPE_VARCHAR>, 9, 9, None, None, 1),
 ('MGR', <cx_Oracle.DbType DB_TYPE_NUMBER>, 5, None, 4, 0, 1),
 ('HIREDATE', <cx_Oracle.DbType DB_TYPE_DATE>, 23, None, None, None, 1),
 ('SAL', <cx_Oracle.DbType DB_TYPE_NUMBER>, 11, None, 7, 2, 1),
 ('COMM', <cx_Oracle.DbType DB_TYPE_NUMBER>, 11, None, 7, 2, 1),
 ('DEPTNO', <cx_Oracle.DbType DB_TYPE_NUMBER>, 3, None, 2, 0, 1)]

In [50]:
[descipt[0] for descipt in cursor.description]

['NO', 'ENAME', 'JOB', 'MGR', 'HIREDATE', 'SAL', 'COMM', 'DEPTNO']

In [51]:
import pandas as pd
emp_df = pd.DataFrame(emps, columns=[descipt[0] for descipt in cursor.description])
emp_df

,NO,ENAME,JOB,MGR,HIREDATE,SAL,COMM,DEPTNO
0,7369,SMITH,CLERK,7902.0,1980-12-17,800.0,NaN,20
1,7499,ALLEN,SALESMAN,7698.0,1981-02-20,1600.0,300.0,30
2,7521,WARD,SALESMAN,7698.0,1981-02-22,1250.0,500.0,30
3,7566,JONES,MANAGER,7839.0,1981-04-02,2975.0,NaN,20
4,7654,MARTIN,SALESMAN,7698.0,1981-09-28,1250.0,1400.0,30
5,7698,BLAKE,MANAGER,7839.0,1981-05-01,2850.0,NaN,30
6,7782,CLARK,MANAGER,7839.0,1981-06-09,2450.0,NaN,10
7,7788,SCOTT,ANALYST,7566.0,1982-12-09,3000.0,NaN,20
8,7839,KING,PRESIDENT,NaN,1981-11-17,5000.0,NaN,10
9,7844,TURNER,SALESMAN,7698.0,1981-09-08,1500.0,0.0,30


In [57]:
# 사용자로부터 검색할 이름을 받아 해당 데이터 출력
sql = "SELECT * FROM EMP WHERE ENAME=(:ename)"
ename = input('검색할 이름?').upper()
cursor.execute(sql, {'ename':ename})
emp = cursor.fetchone() # 결과가 있으면 해당 데이터를 튜플, 결과가 없으면 none으로 도출
if emp :
    columns = [descript[0] for descript in cursor.description]
    df = pd.DataFrame([emp], columns=columns)
    display(df)
else:
    print('해당 이름이 없습니다')

검색할 이름?smith


,EMPNO,ENAME,JOB,MGR,HIREDATE,SAL,COMM,DEPTNO
0,7369,SMITH,CLERK,7902,1980-12-17,800.0,None,20


In [58]:
for col, data in zip(columns, emp):
    print("{}:{}".format(col, data if data is not None else '-'))

EMPNO:7369
ENAME:SMITH
JOB:CLERK
MGR:7902
HIREDATE:1980-12-17 00:00:00
SAL:800.0
COMM:-
DEPTNO:20


In [59]:
cursor.close()
conn.close()

# 3절. MySQL 연결

| 라이브러리 | 특징 | 
| : -- | :-- |
| **mysql-connector-python** | MySQL 공식 커넥터. phthon으로 구현 |
| **pyMySQL** | 커뮤니티에서 만든 서드파트 라이브러리.(경량) phthon으로 구현 널리 쓰임 |

- pip install pyMySQL

In [60]:
%pip install pyMySQL

     ---------------------------------------- 45.7/45.7 kB 2.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pymysql
conn = pymysql.connect